# Phân tích giá vé máy bay với PySpark — Notebook khởi đầu

**Môn:** Tích hợp và phân tích dữ liệu lớn

Notebook này đi qua đủ 6 mốc của pipeline:

| Mốc | Nội dung trong notebook |
|---|---|
| 1. Raw Data | Bộ *Flight Prices* (Kaggle, Expedia 2022, ~82 triệu dòng, ~31 GB CSV) |
| 2. Ingestion | Đọc CSV bằng Spark với schema khai báo sẵn, gọi API thời tiết |
| 3. ETL | Làm sạch, tạo đặc trưng, **tích hợp 4 nguồn** (giá vé + sân bay + ngày lễ + thời tiết) |
| 4. Storage | 3 tầng Parquet: Bronze (thô) → Silver (sạch) → Gold (tổng hợp) |
| 5. Machine Learning | Dự đoán giá vé bằng Spark MLlib (GBT) |
| 6. Visualization | Biểu đồ trả lời các câu hỏi phân tích |

**Chạy ở đâu:** Google Colab (khuyên dùng để bắt đầu) hoặc Databricks. Trên Databricks, đổi `BASE_DIR` sang một đường dẫn Volume.

**Lưu ý:** Lần chạy đầu phải tải ~31 GB và quét toàn bộ file CSV nên mất khá lâu. Từ lần sau notebook đọc lại từ Parquet nên nhanh hơn nhiều.

## 0. Cài đặt và khởi tạo Spark

In [ ]:
%pip install -q holidays kagglehub

In [ ]:
import importlib.util, subprocess, sys

# Colab thường có sẵn pyspark; Databricks luôn có sẵn. Chỉ cài khi thiếu.
if importlib.util.find_spec("pyspark") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyspark"])

from pyspark.sql import SparkSession, functions as F, types as T

# Trên Databricks biến `spark` đã có sẵn nên bỏ qua bước tạo session.
if "spark" not in globals():
    spark = (
        SparkSession.builder.appName("flight-prices")
        .master("local[*]")
        .config("spark.driver.memory", "8g")
        .config("spark.sql.shuffle.partitions", "64")
        .getOrCreate()
    )
print("Spark", spark.version)

## 1. Cấu hình

- `SAMPLE_FRACTION`: tỉ lệ dữ liệu lấy mẫu. Để `0.05` (khoảng 4 triệu dòng) khi làm thử, tăng dần lên `1.0` khi chạy bản chính thức.
- `CSV_PATH`: nếu đã tải sẵn file `itineraries.csv` thì điền đường dẫn vào, notebook sẽ không tải lại.

In [ ]:
import os, glob

BASE_DIR = "/content/flight"      # Databricks: ví dụ "/Volumes/workspace/default/flight"
CSV_PATH = None                   # đường dẫn itineraries.csv nếu đã tải sẵn
SAMPLE_FRACTION = 0.05            # 1.0 = dùng toàn bộ dữ liệu

BRONZE = f"{BASE_DIR}/bronze/itineraries"
SILVER = f"{BASE_DIR}/silver/itineraries"
GOLD = f"{BASE_DIR}/gold"
os.makedirs(BASE_DIR, exist_ok=True)

## 2. Raw Data + Ingestion: tải dữ liệu và nạp vào tầng Bronze

In [ ]:
# Bỏ qua bước tải nếu tầng Bronze đã tồn tại.
if CSV_PATH is None and not os.path.exists(BRONZE):
    import kagglehub
    ds_dir = kagglehub.dataset_download("dilwong/flightprices")
    CSV_PATH = glob.glob(os.path.join(ds_dir, "**", "itineraries.csv"), recursive=True)[0]
print("CSV:", CSV_PATH)

In [ ]:
# Khai báo schema để Spark không phải quét 31 GB chỉ để đoán kiểu dữ liệu.
schema = T.StructType([
    T.StructField("legId", T.StringType()),
    T.StructField("searchDate", T.DateType()),
    T.StructField("flightDate", T.DateType()),
    T.StructField("startingAirport", T.StringType()),
    T.StructField("destinationAirport", T.StringType()),
    T.StructField("fareBasisCode", T.StringType()),
    T.StructField("travelDuration", T.StringType()),
    T.StructField("elapsedDays", T.IntegerType()),
    T.StructField("isBasicEconomy", T.BooleanType()),
    T.StructField("isRefundable", T.BooleanType()),
    T.StructField("isNonStop", T.BooleanType()),
    T.StructField("baseFare", T.DoubleType()),
    T.StructField("totalFare", T.DoubleType()),
    T.StructField("seatsRemaining", T.IntegerType()),
    T.StructField("totalTravelDistance", T.DoubleType()),
    T.StructField("segmentsDepartureTimeEpochSeconds", T.StringType()),
    T.StructField("segmentsDepartureTimeRaw", T.StringType()),
    T.StructField("segmentsArrivalTimeEpochSeconds", T.StringType()),
    T.StructField("segmentsArrivalTimeRaw", T.StringType()),
    T.StructField("segmentsArrivalAirportCode", T.StringType()),
    T.StructField("segmentsDepartureAirportCode", T.StringType()),
    T.StructField("segmentsAirlineName", T.StringType()),
    T.StructField("segmentsAirlineCode", T.StringType()),
    T.StructField("segmentsEquipmentDescription", T.StringType()),
    T.StructField("segmentsDurationInSeconds", T.StringType()),
    T.StructField("segmentsDistance", T.StringType()),
    T.StructField("segmentsCabinCode", T.StringType()),
])

if not os.path.exists(BRONZE):
    raw = spark.read.csv(CSV_PATH, header=True, schema=schema)
    if SAMPLE_FRACTION < 1.0:
        raw = raw.sample(fraction=SAMPLE_FRACTION, seed=42)
    raw.write.mode("overwrite").parquet(BRONZE)

bronze = spark.read.parquet(BRONZE)
print("Số dòng Bronze:", bronze.count())
bronze.select("searchDate", "flightDate", "startingAirport", "destinationAirport",
              "totalFare", "seatsRemaining", "segmentsAirlineName").show(5, truncate=False)

## 3. ETL: làm sạch và tạo đặc trưng (tầng Silver)

Các cột chứa nhiều chặng bay được nối bằng `||` (ví dụ `AA||AA`). Ta tách ra để lấy số chặng và hãng bay của chặng đầu.

Đặc trưng quan trọng nhất là **`days_before`** = số ngày đặt trước (ngày bay − ngày tìm kiếm).

In [ ]:
def first_part(c):
    return F.split(F.col(c), r"\|\|").getItem(0)

def iso_part(pattern):
    # travelDuration có dạng ISO 8601, ví dụ "PT2H29M" hoặc "P1DT3H5M"
    return F.coalesce(F.regexp_extract("travelDuration", pattern, 1).cast("int"), F.lit(0))

silver = (
    bronze
    .filter(F.col("totalFare") > 0)
    .filter(F.col("searchDate").isNotNull() & F.col("flightDate").isNotNull())
    .dropDuplicates(["legId", "searchDate"])
    .withColumn("days_before", F.datediff("flightDate", "searchDate"))
    .withColumn("duration_min",
                iso_part(r"P(\d+)D") * 1440 + iso_part(r"(\d+)H") * 60 + iso_part(r"(\d+)M"))
    .withColumn("num_stops", F.size(F.split("segmentsAirlineCode", r"\|\|")) - 1)
    .withColumn("airline_code", first_part("segmentsAirlineCode"))
    .withColumn("airline_name", first_part("segmentsAirlineName"))
    # Giờ khởi hành theo giờ địa phương, lấy từ chuỗi "2022-04-17T12:57:00.000-04:00"
    .withColumn("dep_hour", F.substring(first_part("segmentsDepartureTimeRaw"), 12, 2).cast("int"))
    .withColumn("day_of_week", F.dayofweek("flightDate"))          # 1 = Chủ nhật ... 7 = Thứ bảy
    .withColumn("is_weekend", F.col("day_of_week").isin(1, 7).cast("int"))
    .withColumn("route", F.concat_ws("-", "startingAirport", "destinationAirport"))
    .filter(F.col("days_before") >= 0)
    .select("legId", "searchDate", "flightDate", "startingAirport", "destinationAirport", "route",
            "airline_code", "airline_name", "isBasicEconomy", "isRefundable", "isNonStop",
            "baseFare", "totalFare", "seatsRemaining", "totalTravelDistance",
            "days_before", "duration_min", "num_stops", "dep_hour", "day_of_week", "is_weekend")
)

## 4. Tích hợp dữ liệu: thêm 3 nguồn ngoài

| Nguồn | Khóa ghép | Cột thêm vào |
|---|---|---|
| OpenFlights (danh mục sân bay) | mã IATA | thành phố, tọa độ |
| Thư viện `holidays` (ngày lễ Mỹ) | ngày bay | `is_holiday` |
| Open-Meteo (thời tiết lịch sử) | sân bay đi + ngày bay | nhiệt độ, lượng mưa, gió |

Ba bảng này rất nhỏ nên dùng **broadcast join** để không phải shuffle bảng lớn.

In [ ]:
import pandas as pd

# --- Nguồn 2: danh mục sân bay (OpenFlights) ---
AIRPORTS_URL = "https://raw.githubusercontent.com/jpatokal/openflights/master/data/airports.dat"
ap_cols = ["id", "name", "city", "country", "iata", "icao", "lat", "lon",
           "alt", "tz", "dst", "tzdb", "type", "source"]
ap = pd.read_csv(AIRPORTS_URL, header=None, names=ap_cols, na_values="\\N")

used_codes = {r[0] for r in silver.select("startingAirport").distinct().collect()} | \
             {r[0] for r in silver.select("destinationAirport").distinct().collect()}
ap_used = ap[ap["iata"].isin(used_codes)][["iata", "city", "lat", "lon"]].reset_index(drop=True)
print(len(ap_used), "sân bay:", sorted(ap_used["iata"]))

airports = spark.createDataFrame(ap_used)

In [ ]:
# --- Nguồn 3: ngày lễ ---
import holidays as holidays_lib

dmin, dmax = silver.agg(F.min("flightDate"), F.max("flightDate")).first()
us_holidays = holidays_lib.US(years=list(range(dmin.year, dmax.year + 1)))
hol = spark.createDataFrame(pd.DataFrame({
    "flightDate": list(us_holidays.keys()),
    "holiday_name": list(us_holidays.values()),
}))
print("Khoảng ngày bay:", dmin, "→", dmax)

In [ ]:
# --- Nguồn 4: thời tiết lịch sử tại sân bay đi (Open-Meteo, miễn phí, không cần API key) ---
import requests

weather = None
try:
    frames = []
    for _, a in ap_used.iterrows():
        resp = requests.get(
            "https://archive-api.open-meteo.com/v1/archive",
            params={
                "latitude": a["lat"], "longitude": a["lon"],
                "start_date": str(dmin), "end_date": str(dmax),
                "daily": "temperature_2m_mean,precipitation_sum,wind_speed_10m_max",
                "timezone": "auto",
            },
            timeout=60,
        )
        resp.raise_for_status()
        d = resp.json()["daily"]
        frames.append(pd.DataFrame({
            "startingAirport": a["iata"],
            "flightDate": pd.to_datetime(d["time"]).date,
            "temp_mean": pd.to_numeric(d["temperature_2m_mean"], errors="coerce"),
            "precip_mm": pd.to_numeric(d["precipitation_sum"], errors="coerce"),
            "wind_max": pd.to_numeric(d["wind_speed_10m_max"], errors="coerce"),
        }))
    weather = spark.createDataFrame(pd.concat(frames, ignore_index=True))
    print("Số dòng thời tiết:", weather.count())
except Exception as e:
    print("Không lấy được thời tiết, bỏ qua nguồn này:", e)

In [ ]:
# --- Ghép tất cả vào bảng Silver ---
origin = airports.select(F.col("iata").alias("startingAirport"), F.col("city").alias("origin_city"))
dest = airports.select(F.col("iata").alias("destinationAirport"), F.col("city").alias("dest_city"))

silver_i = (
    silver
    .join(F.broadcast(origin), "startingAirport", "left")
    .join(F.broadcast(dest), "destinationAirport", "left")
    .join(F.broadcast(hol), "flightDate", "left")
    .withColumn("is_holiday", F.col("holiday_name").isNotNull().cast("int"))
)
if weather is not None:
    silver_i = silver_i.join(F.broadcast(weather), ["startingAirport", "flightDate"], "left")
else:
    for c in ["temp_mean", "precip_mm", "wind_max"]:
        silver_i = silver_i.withColumn(c, F.lit(None).cast("double"))

silver_i.write.mode("overwrite").parquet(SILVER)
silver_i = spark.read.parquet(SILVER)
print("Số dòng Silver:", silver_i.count())
silver_i.printSchema()

## 5. Storage: các bảng tổng hợp (tầng Gold)

Mỗi bảng Gold trả lời một câu hỏi phân tích. Các bảng này nhỏ, dùng trực tiếp cho dashboard (Power BI, Streamlit...).

In [ ]:
def agg_price(df, *keys):
    return (df.groupBy(*keys)
              .agg(F.count("*").alias("n"),
                   F.round(F.avg("totalFare"), 2).alias("avg_fare"),
                   F.round(F.percentile_approx("totalFare", 0.5), 2).alias("median_fare"))
              .orderBy(*keys))

gold = {
    "by_days_before": agg_price(silver_i.filter(F.col("days_before") <= 60), "days_before"),
    "by_day_of_week": agg_price(silver_i, "day_of_week"),
    "by_dep_hour": agg_price(silver_i.filter(F.col("dep_hour").isNotNull()), "dep_hour"),
    "by_airline": agg_price(silver_i, "airline_name"),
    "by_stops": agg_price(silver_i, "num_stops"),
    "by_seats": agg_price(silver_i.filter(F.col("seatsRemaining").isNotNull()), "seatsRemaining"),
    "by_holiday": agg_price(silver_i, "is_holiday"),
    "by_route": agg_price(silver_i, "route"),
}

pdf = {}
for name, df in gold.items():
    df.write.mode("overwrite").parquet(f"{GOLD}/{name}")
    pdf[name] = df.toPandas()
    print(name, len(pdf[name]), "dòng")

## 6. Visualization: trả lời các câu hỏi phân tích

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(18, 9))

d = pdf["by_days_before"]
axes[0, 0].plot(d["days_before"], d["median_fare"], marker="o", ms=3)
axes[0, 0].set(title="Nên đặt vé trước bao nhiêu ngày?", xlabel="Số ngày đặt trước", ylabel="Giá trung vị (USD)")
axes[0, 0].invert_xaxis()

d = pdf["by_day_of_week"]
labels = ["CN", "T2", "T3", "T4", "T5", "T6", "T7"]
axes[0, 1].bar([labels[i - 1] for i in d["day_of_week"]], d["median_fare"])
axes[0, 1].set(title="Giá theo thứ trong tuần (ngày bay)", ylabel="Giá trung vị (USD)")

d = pdf["by_dep_hour"]
axes[0, 2].bar(d["dep_hour"], d["median_fare"])
axes[0, 2].set(title="Giá theo giờ khởi hành", xlabel="Giờ (địa phương)", ylabel="Giá trung vị (USD)")

d = pdf["by_airline"].sort_values("median_fare")
axes[1, 0].barh(d["airline_name"], d["median_fare"])
axes[1, 0].set(title="Giá theo hãng bay (chặng đầu)", xlabel="Giá trung vị (USD)")

d = pdf["by_stops"]
axes[1, 1].bar(d["num_stops"].astype(str), d["median_fare"])
axes[1, 1].set(title="Bay thẳng và nối chuyến", xlabel="Số điểm dừng", ylabel="Giá trung vị (USD)")

d = pdf["by_seats"]
axes[1, 2].plot(d["seatsRemaining"], d["median_fare"], marker="o")
axes[1, 2].set(title="Giá theo số ghế còn lại", xlabel="Số ghế còn", ylabel="Giá trung vị (USD)")

plt.tight_layout()
plt.show()

In [ ]:
# Tuyến đắt nhất / rẻ nhất và ảnh hưởng của ngày lễ
routes = pdf["by_route"].sort_values("median_fare")
print("5 tuyến rẻ nhất:\n", routes.head(5).to_string(index=False))
print("\n5 tuyến đắt nhất:\n", routes.tail(5).to_string(index=False))
print("\nNgày thường (0) và ngày lễ (1):\n", pdf["by_holiday"].to_string(index=False))

## 7. Machine Learning: dự đoán giá vé

Mô hình **Gradient-Boosted Trees** của Spark MLlib, huấn luyện trên một mẫu để chạy nhanh. Đánh giá bằng RMSE, MAE và R².

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.regression import GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator

ML_ROWS = 500_000
cat_cols = ["airline_code", "startingAirport", "destinationAirport"]
num_cols = ["days_before", "duration_min", "num_stops", "seatsRemaining", "totalTravelDistance",
            "dep_hour", "day_of_week", "is_weekend", "is_holiday", "basic_economy", "refundable"]

ml_df = (
    silver_i
    .withColumn("basic_economy", F.col("isBasicEconomy").cast("int"))
    .withColumn("refundable", F.col("isRefundable").cast("int"))
    .select("totalFare", *cat_cols, *num_cols)
    .dropna()
)
total = ml_df.count()
ml_df = ml_df.sample(fraction=min(1.0, ML_ROWS / total), seed=42).cache()
train, test = ml_df.randomSplit([0.8, 0.2], seed=42)
print("Train:", train.count(), "| Test:", test.count())

indexers = [StringIndexer(inputCol=c, outputCol=c + "_idx", handleInvalid="keep") for c in cat_cols]
feature_cols = [c + "_idx" for c in cat_cols] + num_cols
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
gbt = GBTRegressor(featuresCol="features", labelCol="totalFare", maxIter=50, maxDepth=6, maxBins=64, seed=42)

model = Pipeline(stages=indexers + [assembler, gbt]).fit(train)
pred = model.transform(test)

for metric in ["rmse", "mae", "r2"]:
    value = RegressionEvaluator(labelCol="totalFare", predictionCol="prediction", metricName=metric).evaluate(pred)
    print(f"{metric.upper()}: {value:.3f}")

In [ ]:
# Yếu tố nào ảnh hưởng đến giá nhiều nhất?
imp = pd.Series(model.stages[-1].featureImportances.toArray(), index=feature_cols).sort_values()
imp.plot.barh(figsize=(8, 6), title="Mức độ quan trọng của đặc trưng (GBT)")
plt.tight_layout()
plt.show()

pred.select("totalFare", F.round("prediction", 2).alias("prediction"),
            "airline_code", "startingAirport", "destinationAirport", "days_before").show(10)

## 8. Việc tiếp theo cho nhóm

1. **Tăng dữ liệu:** nâng `SAMPLE_FRACTION` lên `1.0`, xóa thư mục Bronze cũ rồi chạy lại.
2. **Thêm thời tiết vào mô hình:** đưa `temp_mean`, `precip_mm`, `wind_max` vào `num_cols` và so sánh R² trước/sau — đây là bằng chứng cho giá trị của việc tích hợp dữ liệu.
3. **Thêm nguồn:** giá nhiên liệu bay (EIA) ghép theo tuần.
4. **Phân tích sâu hơn:** giá của cùng một chuyến bay (`legId`) thay đổi thế nào qua các ngày tìm kiếm; phát hiện vé rẻ bất thường theo từng tuyến.
5. **So sánh mô hình:** Linear Regression, Random Forest, GBT; tinh chỉnh bằng `CrossValidator`.
6. **Dashboard:** đọc các bảng trong thư mục Gold bằng Power BI hoặc Streamlit.
7. **Orchestration:** tách notebook thành các bước (ingest → silver → gold → train) và lập lịch bằng Airflow hoặc Databricks Jobs.